# LAB | Relevance Scoring and Rerankers
**Advanced RAG with metadata filtering, LLM relevance scoring, and Cohere reranking.**

## Step 0 — Install Dependencies
Run once, then restart the kernel.

In [1]:
!pip install openai numpy pypdf2 tiktoken python-dotenv -q
!pip install langchain langchain-text-splitters langchain-core -q
!pip install cohere sentence-transformers -q
!pip install pydub imageio[ffmpeg] openai-whisper -q


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\eliza\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\eliza\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\eliza\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\eliza\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## Step 1 — Imports & API Setup

In [2]:
import os
import numpy as np
import PyPDF2
import tiktoken
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from typing import List, Dict
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load API keys
load_dotenv(
    r"c:\Users\eliza\OneDrive\Documents\W8_LAB_RAG with OpenAI_native Implementation\.env",
    override=True
)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

print("OpenAI key:", OPENAI_API_KEY[:8] + "..." if OPENAI_API_KEY else "❌ MISSING")
print("Cohere key:", COHERE_API_KEY[:8] + "..." if COHERE_API_KEY else "❌ MISSING")

# OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Cohere client (optional)
try:
    import cohere
    co = cohere.Client(COHERE_API_KEY)
    COHERE_AVAILABLE = True
    print("✅ Cohere ready")
except Exception as e:
    co = None
    COHERE_AVAILABLE = False
    print(f"⚠️  Cohere not available: {e}")

# Verify OpenAI
try:
    client.models.list()
    print("✅ OpenAI ready")
except Exception as e:
    print(f"❌ OpenAI error: {e}")

OpenAI key: sk-svcac...
Cohere key: w3LlZ8np...
✅ Cohere ready
✅ OpenAI ready


## Step 1b — Load PDF

In [3]:
def load_pdf(pdf_path: str) -> str:
    reader = PyPDF2.PdfReader(pdf_path)
    text = "\n\n".join(
        page.extract_text() for page in reader.pages if page.extract_text()
    )
    print(f"✅ PDF loaded: {len(text):,} chars, {len(reader.pages)} pages")
    return text

pdf_path = r"C:\Users\eliza\OneDrive\Documents\W8_LAB_RAG with OpenAI_native Implementation\Document\ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf"

if not Path(pdf_path).exists():
    print("❌ PDF not found — searching...")
    for p in Path(r"C:\Users\eliza\OneDrive\Documents").rglob("*.pdf"):
        print(p)
else:
    PDF_TEXT = load_pdf(pdf_path)

✅ PDF loaded: 158,118 chars, 41 pages


## Step 1c — Load Podcast via Whisper

In [5]:
import urllib.request, zipfile, imageio_ffmpeg, os
import whisper
import numpy as np
from pydub import AudioSegment
from pydub import utils as pydub_utils
from pathlib import Path

# ── Point pydub explicitly to bundled ffmpeg + ffprobe ──────
ffmpeg_exe   = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_dir   = os.path.dirname(ffmpeg_exe)
ffprobe_path = os.path.join(ffmpeg_dir, "ffprobe.exe")

# Download ffprobe if missing
if not os.path.exists(ffprobe_path):
    print("⏳ Downloading ffprobe...")
    zip_path = os.path.join(ffmpeg_dir, "ffprobe.zip")
    urllib.request.urlretrieve(
        "https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-win64-gpl.zip",
        zip_path
    )
    with zipfile.ZipFile(zip_path, "r") as z:
        for name in z.namelist():
            if name.endswith("ffprobe.exe"):
                with z.open(name) as src, open(ffprobe_path, "wb") as dst:
                    dst.write(src.read())
                print("✅ ffprobe.exe installed:", ffprobe_path)
                break
    os.remove(zip_path)

# ✅ Override pydub's prober directly — this is the key fix
AudioSegment.converter      = ffmpeg_exe
AudioSegment.ffmpeg         = ffmpeg_exe
AudioSegment.ffprobe        = ffprobe_path
pydub_utils.get_prober_name = lambda: ffprobe_path  # ← forces pydub to use bundled ffprobe

print("ffmpeg exists:", os.path.exists(ffmpeg_exe))
print("ffprobe exists:", os.path.exists(ffprobe_path))

# ── Transcribe ───────────────────────────────────────────────
audio_path = Path(r"C:\Users\eliza\OneDrive\Documents\W8_LAB_RAG with OpenAI_native Implementation\Document\The_Blueprint_For_Trustworthy_AI.m4a")

if not audio_path.exists():
    print("❌ Audio not found — searching...")
    candidates = list(Path(r"C:\Users\eliza\OneDrive\Documents").rglob("*.m4a"))
    if candidates:
        audio_path = candidates[0]
        print(f"✅ Found: {audio_path}")

if audio_path.exists():
    audio   = AudioSegment.from_file(str(audio_path), format="m4a")
    audio   = audio.set_frame_rate(16000).set_channels(1)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    samples /= np.iinfo(audio.array_type).max
    print("⏳ Transcribing podcast...")
    whisper_model = whisper.load_model("base")
    PODCAST_TEXT  = whisper_model.transcribe(samples)["text"]
    print(f"✅ Podcast transcript: {len(PODCAST_TEXT):,} chars")
else:
    PODCAST_TEXT = ""
    print("⚠️  No audio found — PODCAST_TEXT is empty")

ffmpeg exists: True
ffprobe exists: True
❌ Audio not found — searching...
✅ Found: C:\Users\eliza\OneDrive\Documents\W8_LAB_RAG_Relevance Scoring and Rerankers\Document\The_Blueprint_For_Trustworthy_AI.m4a
⏳ Transcribing podcast...


c:\Users\eliza\anaconda3\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Podcast transcript: 16,399 chars


## Step 1d — Chunk with Rich Metadata
Rich metadata (e.g. `source_type`) enables filtering in Step 5.

In [6]:
def token_count(text: str) -> int:
    enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))

def chunk_and_attach_metadata(
    text: str,
    source: str,
    source_type: str,       # "podcast" | "pdf"
    strategy: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
) -> List[Dict]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_text(text)
    docs   = []
    for i, chunk in enumerate(chunks):
        docs.append({
            "page_content": chunk,
            "metadata": {
                "source":       source,
                "source_type":  source_type,
                "strategy":     strategy,
                "chunk_index":  i,
                "total_chunks": len(chunks),
                "char_count":   len(chunk),
                "token_count":  token_count(chunk),
            }
        })
    print(f"✅ {source}: {len(docs)} chunks")
    return docs

podcast_docs = chunk_and_attach_metadata(
    PODCAST_TEXT, source="podcast_transcript",
    source_type="podcast", strategy="recursive_1000_200"
) if PODCAST_TEXT else []

pdf_docs = chunk_and_attach_metadata(
    PDF_TEXT, source="trustworthy_ai_pdf",
    source_type="pdf", strategy="recursive_1000_200"
)

all_docs = podcast_docs + pdf_docs
print(f"\n✅ Total chunks: {len(all_docs)}")
print("Sample metadata:", all_docs[0]["metadata"])

✅ podcast_transcript: 21 chunks
✅ trustworthy_ai_pdf: 206 chunks

✅ Total chunks: 227
Sample metadata: {'source': 'podcast_transcript', 'source_type': 'podcast', 'strategy': 'recursive_1000_200', 'chunk_index': 0, 'total_chunks': 21, 'char_count': 988, 'token_count': 217}


## Step 2 — Generate Embeddings & Baseline Retrieval

In [7]:
def get_embeddings_batch(
    texts: List[str],
    model: str = "text-embedding-3-small",
    batch_size: int = 100,
) -> List[List[float]]:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch    = texts[i:i + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        all_embeddings.extend([item.embedding for item in response.data])
        print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks")
    return all_embeddings

print("⏳ Generating embeddings...")
texts      = [doc["page_content"] for doc in all_docs]
embeddings = get_embeddings_batch(texts)

for doc, emb in zip(all_docs, embeddings):
    doc["embedding"] = emb

print(f"✅ {len(embeddings)} embeddings ready (dim={len(embeddings[0])})")

⏳ Generating embeddings...
  Embedded 100/227 chunks
  Embedded 200/227 chunks
  Embedded 227/227 chunks
✅ 227 embeddings ready (dim=1536)


In [8]:
def cosine_similarity(a: List[float], b: List[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve_baseline(
    query: str,
    docs: List[Dict],
    top_k: int = 10,
    source_type: str = None,
    model: str = "text-embedding-3-small",
) -> List[Dict]:
    """Baseline: embed query → cosine similarity → top_k."""
    response  = client.embeddings.create(model=model, input=[query])
    query_emb = response.data[0].embedding

    candidates = [d for d in docs if d["metadata"]["source_type"] == source_type] \
                 if source_type else docs

    scored = sorted(
        [{**doc, "similarity_score": cosine_similarity(query_emb, doc["embedding"])}
         for doc in candidates],
        key=lambda x: x["similarity_score"], reverse=True
    )
    return scored[:top_k]

# Quick baseline test
q = "What are the core principles of trustworthy AI?"
baseline = retrieve_baseline(q, all_docs, top_k=5)
print(f"✅ Baseline top score: {baseline[0]['similarity_score']:.4f}")
print(f"   Source: {baseline[0]['metadata']['source']}")
print(f"   Preview: {baseline[0]['page_content'][:200]}...")

✅ Baseline top score: 0.7455
   Source: trustworthy_ai_pdf
   Preview: technology, are trustworthy. When drafting these G uidelines, Trustworthy  AI has, therefore, been our foundational 
ambition.  
Trustworthy  AI has thre e components: (1) it should be l awful, ensuri...


## Step 3 — LLM Relevance Scoring (Advanced)
Use GPT to score each chunk's relevance, then combine with similarity score.

In [9]:
def llm_relevance_score(query: str, chunk: str, model: str = "gpt-4o-mini") -> float:
    """Ask GPT to score relevance 0.0-1.0."""
    prompt = f"""Score how relevant the passage is to the query.
Return ONLY a number between 0.0 (not relevant) and 1.0 (highly relevant). Nothing else.

Query: {query}
Passage: {chunk[:500]}
Score:"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=5,
    )
    try:
        return float(response.choices[0].message.content.strip())
    except:
        return 0.0

def retrieve_llm_reranked(
    query: str,
    docs: List[Dict],
    top_k: int = 5,
    initial_k: int = 20,
    source_type: str = None,
) -> List[Dict]:
    """
    1. Retrieve initial_k via cosine similarity
    2. Score each with LLM (0-1)
    3. Combined score = 0.5 * similarity + 0.5 * llm_score
    4. Return top_k reranked
    """
    candidates = retrieve_baseline(query, docs, top_k=initial_k, source_type=source_type)
    print(f"⏳ LLM scoring {len(candidates)} candidates...")
    for c in candidates:
        c["llm_score"]      = llm_relevance_score(query, c["page_content"])
        c["combined_score"] = 0.5 * c["similarity_score"] + 0.5 * c["llm_score"]
    return sorted(candidates, key=lambda x: x["combined_score"], reverse=True)[:top_k]

# Test LLM reranking
llm_results = retrieve_llm_reranked(q, all_docs, top_k=5, initial_k=20)
print(f"\n✅ LLM reranked top score: {llm_results[0]['combined_score']:.4f}")
print(f"   similarity={llm_results[0]['similarity_score']:.4f} | llm={llm_results[0]['llm_score']:.4f}")

⏳ LLM scoring 20 candidates...

✅ LLM reranked top score: 0.8728
   similarity=0.7455 | llm=1.0000


## Step 4 — Cohere Reranker (Advanced)
Use Cohere Rerank API to rerank the initial retrieval results.

In [10]:
def retrieve_cohere_reranked(
    query: str,
    docs: List[Dict],
    top_k: int = 5,
    initial_k: int = 20,
    source_type: str = None,
) -> List[Dict]:
    """
    1. Retrieve initial_k via cosine similarity
    2. Rerank with Cohere Rerank API
    3. Return top_k reranked
    """
    if not COHERE_AVAILABLE:
        print("⚠️  Cohere not available — falling back to baseline")
        return retrieve_baseline(query, docs, top_k=top_k, source_type=source_type)

    candidates = retrieve_baseline(query, docs, top_k=initial_k, source_type=source_type)
    texts      = [c["page_content"] for c in candidates]

    rerank_resp = co.rerank(
        query=query,
        documents=texts,
        top_n=top_k,
        model="rerank-english-v3.0",
    )

    reranked = []
    for r in rerank_resp.results:
        doc = candidates[r.index].copy()
        doc["cohere_score"]   = r.relevance_score
        doc["combined_score"] = 0.5 * doc["similarity_score"] + 0.5 * r.relevance_score
        reranked.append(doc)
    return reranked

# Test Cohere reranking
if COHERE_AVAILABLE:
    cohere_results = retrieve_cohere_reranked(q, all_docs, top_k=5, initial_k=20)
    print(f"✅ Cohere reranked top score: {cohere_results[0]['cohere_score']:.4f}")
else:
    print("⚠️  Cohere not available — skipping test")

✅ Cohere reranked top score: 0.9983


## Step 5 — Metadata Filtering
Filter results by `source_type` to query only PDF or only podcast chunks.

In [11]:
print("── Metadata filter: PDF only ──")
pdf_only = retrieve_baseline(q, all_docs, top_k=3, source_type="pdf")
for r in pdf_only:
    print(f"  [{r['metadata']['source_type']}] score={r['similarity_score']:.4f} | {r['page_content'][:100]}...")

print("\n── Metadata filter: Podcast only ──")
pod_only = retrieve_baseline(q, all_docs, top_k=3, source_type="podcast")
for r in pod_only:
    print(f"  [{r['metadata']['source_type']}] score={r['similarity_score']:.4f} | {r['page_content'][:100]}...")

── Metadata filter: PDF only ──
  [pdf] score=0.7455 | technology, are trustworthy. When drafting these G uidelines, Trustworthy  AI has, therefore, been o...
  [pdf] score=0.7386 | entire life cycle.  
Trustworthy  AI has three components , which should be met throughout the syste...
  [pdf] score=0.7218 | 9 
 I. Chapter I: Foundations of Trustworthy  AI  
This Chapter sets out the foundations of Trustwor...

── Metadata filter: Podcast only ──
  [podcast] score=0.6648 | . So why are we blowing the dust off this specific report? Because it's not dusty. It's the foundati...
  [podcast] score=0.6548 | So imagine for a second you're driving across a, I don't know, a massive suspension bridge. Okay. Yo...
  [podcast] score=0.6310 | . It's such a distinct shift from, you know, optimizing efficiency or maximizing engagement. It is. ...


## Step 6 — Full RAG Pipeline with Reranking
Switch reranker with **one argument** — `baseline`, `llm`, or `cohere`.

In [12]:
def rag_query(
    question: str,
    docs: List[Dict],
    reranker: str = "baseline",     # "baseline" | "llm" | "cohere"
    top_k: int = 5,
    source_type: str = None,
    model: str = "gpt-4o-mini",
) -> Dict:
    """Complete RAG pipeline. Switch reranker with one argument."""

    # 1-line change to switch reranking strategy
    if reranker == "llm":
        chunks = retrieve_llm_reranked(question, docs, top_k=top_k, source_type=source_type)
    elif reranker == "cohere":
        chunks = retrieve_cohere_reranked(question, docs, top_k=top_k, source_type=source_type)
    else:
        chunks = retrieve_baseline(question, docs, top_k=top_k, source_type=source_type)

    # Build context with citations
    context = "\n\n---\n\n".join([
        f"[Source {i+1} | {c['metadata']['source']} | chunk {c['metadata']['chunk_index']}]\n{c['page_content']}"
        for i, c in enumerate(chunks)
    ])

    # Generate answer
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": (
                "Answer questions using only the provided context. "
                "Cite sources as [Source N]. If the answer is not in the context, say so."
            )},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"},
        ],
        temperature=0.2,
    )

    score_key = "combined_score" if reranker != "baseline" else "similarity_score"
    return {
        "question":    question,
        "reranker":    reranker,
        "answer":      response.choices[0].message.content,
        "sources":     [{"source":  c["metadata"]["source"],
                         "chunk":   c["metadata"]["chunk_index"],
                         "score":   round(c.get(score_key, 0), 4),
                         "preview": c["page_content"][:150]} for c in chunks],
        "tokens_used": response.usage.total_tokens,
    }

# Quick test
result = rag_query(q, all_docs, reranker="baseline", top_k=5)
print(f"✅ RAG pipeline working")
print(f"Answer preview: {result['answer'][:300]}...")

✅ RAG pipeline working
Answer preview: The core principles of trustworthy AI are based on three components: 

1. **Lawfulness**: Ensuring compliance with all applicable laws and regulations.
2. **Ethicality**: Adhering to ethical principles and values.
3. **Robustness**: Being robust from both a technical and social perspective to preven...


## Step 7 — Evaluate Performance
Compare all reranking approaches side by side. Manually assess answer quality.

In [13]:
test_questions = [
    "What are the core principles of trustworthy AI?",
    "How does the EU AI Act classify high-risk AI systems?",
    "What role does human oversight play in AI governance?",
    "What techniques ensure fairness in AI systems?",
]

rerankers = ["baseline", "llm"]
if COHERE_AVAILABLE:
    rerankers.append("cohere")

print("=" * 65)
print("STEP 7: EVALUATION — Compare Retrieval Approaches")
print("=" * 65)

for question in test_questions:
    print(f"\n🔵 Question: {question}")
    print("-" * 65)
    for method in rerankers:
        result = rag_query(question, all_docs, reranker=method, top_k=5)
        print(f"\n[{method.upper()}]")
        print(f"Answer: {result['answer'][:400]}")
        print(f"Top source: {result['sources'][0]['source']} | score={result['sources'][0]['score']}")
        print(f"Tokens used: {result['tokens_used']}")
    print("=" * 65)

print("\n✅ Evaluation complete!")
print("Manually compare answers above to assess retrieval quality improvement.")
print("\nKey questions to consider:")
print("  • Does reranking surface more relevant chunks?")
print("  • Are answers more precise with LLM/Cohere reranking?")
print("  • Which approach works best for legal/policy questions?")

STEP 7: EVALUATION — Compare Retrieval Approaches

🔵 Question: What are the core principles of trustworthy AI?
-----------------------------------------------------------------

[BASELINE]
Answer: The core principles of trustworthy AI are based on three components: 

1. **Lawfulness** - ensuring compliance with all applicable laws and regulations.
2. **Ethicality** - ensuring adherence to ethical principles and values.
3. **Robustness** - being robust from both a technical and social perspective to prevent unintentional harm, even with good intentions.

These components are necessary but no
Top source: trustworthy_ai_pdf | score=0.7455
Tokens used: 1251
⏳ LLM scoring 20 candidates...

[LLM]
Answer: The core principles of trustworthy AI are:

1. **Lawfulness**: It should comply with all applicable laws and regulations.
2. **Ethical**: It should adhere to ethical principles and values.
3. **Robustness**: It should be robust from both a technical and social perspective, ensuring that AI s